# 03i — Calibración de probabilidades

Sobre el modelo CatBoost tuneado (03g): genera reliability diagram, calcula
Brier score y ECE, compara Platt scaling vs Isotonic regression.

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datetime import datetime

RANDOM_STATE = 42
TARGETS = ['churn_14d', 'churn_30d']

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03i_calibration'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'

def expected_calibration_error(y_true, y_proba, n_bins=10):
    """ECE: media ponderada de |accuracy - confidence| por bin."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_proba >= bins[i]) & (y_proba < bins[i+1] if i < n_bins-1 else y_proba <= bins[i+1])
        if mask.sum() == 0: continue
        acc = y_true[mask].mean()
        conf = y_proba[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

In [2]:
# [EXEC] Cargar datos y splits
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

X_full = master.drop(columns=['churn_14d', 'churn_30d', 'user_id'])
train_mask = splits_df['split'].values == 'train'
val_mask = splits_df['split'].values == 'val'
test_mask = splits_df['split'].values == 'test'
X_train, X_val, X_test = X_full[train_mask], X_full[val_mask], X_full[test_mask]

In [3]:
# [EXEC] Calibrar y evaluar
calibration_metrics = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, target in enumerate(TARGETS):
    print(f'\n=== Calibración {target} ===')
    y = master[target].astype(int)
    y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

    # Cargar modelo tuneado
    model = CatBoostClassifier()
    model.load_model(str(DATA_MODELS / f'catboost_tuned_{target}_v3.cbm'))

    # Probabilidades raw sobre val (para fit) y test (para eval)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    proba_val = model.predict_proba(val_pool)[:, 1]
    proba_test_raw = model.predict_proba(test_pool)[:, 1]

    # Uncalibrated
    bs_raw = float(brier_score_loss(y_test, proba_test_raw))
    ll_raw = float(log_loss(y_test, proba_test_raw))
    ece_raw = expected_calibration_error(y_test.values, proba_test_raw)
    calibration_metrics.append({'target': target, 'method': 'uncalibrated', 'brier': bs_raw, 'ECE': ece_raw, 'log_loss': ll_raw})

    # Platt scaling: fit logistic regression sobre val
    platt = LogisticRegression(max_iter=1000)
    platt.fit(proba_val.reshape(-1, 1), y_val)
    proba_test_platt = platt.predict_proba(proba_test_raw.reshape(-1, 1))[:, 1]
    bs_platt = float(brier_score_loss(y_test, proba_test_platt))
    ll_platt = float(log_loss(y_test, proba_test_platt))
    ece_platt = expected_calibration_error(y_test.values, proba_test_platt)
    calibration_metrics.append({'target': target, 'method': 'platt', 'brier': bs_platt, 'ECE': ece_platt, 'log_loss': ll_platt})

    # Isotonic regression
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(proba_val, y_val)
    proba_test_iso = iso.predict(proba_test_raw)
    bs_iso = float(brier_score_loss(y_test, proba_test_iso))
    ll_iso = float(log_loss(y_test, np.clip(proba_test_iso, 1e-9, 1-1e-9)))
    ece_iso = expected_calibration_error(y_test.values, proba_test_iso)
    calibration_metrics.append({'target': target, 'method': 'isotonic', 'brier': bs_iso, 'ECE': ece_iso, 'log_loss': ll_iso})

    print(f'  uncalibrated: brier={bs_raw:.4f} | ECE={ece_raw:.4f} | LL={ll_raw:.4f}')
    print(f'  platt:        brier={bs_platt:.4f} | ECE={ece_platt:.4f} | LL={ll_platt:.4f}')
    print(f'  isotonic:     brier={bs_iso:.4f} | ECE={ece_iso:.4f} | LL={ll_iso:.4f}')

    # Reliability diagram
    ax = axes[i]
    for proba, label, color in [(proba_test_raw, 'Uncalibrated', 'C0'),
                                  (proba_test_platt, 'Platt', 'C1'),
                                  (proba_test_iso, 'Isotonic', 'C2')]:
        prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10)
        ax.plot(prob_pred, prob_true, marker='o', label=label, color=color)
    ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='Perfect')
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Observed frequency')
    ax.set_title(f'Reliability diagram — {target}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'reliability_diagrams.png', dpi=120, bbox_inches='tight')
plt.close()

cm_df = pd.DataFrame(calibration_metrics)
cm_df.to_csv(REPORT_DIR / 'calibration_metrics.csv', index=False)
print('\n=== Tabla final ===')
print(cm_df.round(4).to_string(index=False))


=== Calibración churn_14d ===
  uncalibrated: brier=0.1305 | ECE=0.0134 | LL=0.4168
  platt:        brier=0.1307 | ECE=0.0209 | LL=0.4193
  isotonic:     brier=0.1308 | ECE=0.0168 | LL=0.4260

=== Calibración churn_30d ===
  uncalibrated: brier=0.1772 | ECE=0.0230 | LL=0.5242
  platt:        brier=0.1781 | ECE=0.0257 | LL=0.5294
  isotonic:     brier=0.1770 | ECE=0.0149 | LL=0.5319

=== Tabla final ===
   target       method  brier    ECE  log_loss
churn_14d uncalibrated 0.1305 0.0134    0.4168
churn_14d        platt 0.1307 0.0209    0.4193
churn_14d     isotonic 0.1308 0.0168    0.4260
churn_30d uncalibrated 0.1772 0.0230    0.5242
churn_30d        platt 0.1781 0.0257    0.5294
churn_30d     isotonic 0.1770 0.0149    0.5319


In [4]:
# [REPORT] calibration_summary.md
lines = [
    '# Calibración de probabilidades — CatBoost tuneado',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Calibradores ajustados sobre VAL set, evaluados sobre TEST.**',
    '',
    '## Métricas',
    '',
    '| target | method | Brier ↓ | ECE ↓ | log_loss ↓ |',
    '|---|---|---:|---:|---:|',
]
for _, row in cm_df.iterrows():
    lines.append(f'| {row["target"]} | {row["method"]} | {row["brier"]:.4f} | {row["ECE"]:.4f} | {row["log_loss"]:.4f} |')

lines += ['', '## Recomendación', '']
for target in TARGETS:
    sub = cm_df[cm_df['target']==target].sort_values('brier').iloc[0]
    lines.append(f'- **{target}**: usar `{sub["method"]}` (Brier={sub["brier"]:.4f}, ECE={sub["ECE"]:.4f})')

lines += [
    '',
    '## Notas',
    '',
    '- **Brier score**: error cuadrático medio sobre las probabilidades. Menor = mejor.',
    '- **ECE** (Expected Calibration Error): media ponderada de |accuracy − confianza| por bin. Menor = mejor calibrado.',
    '- **Platt scaling**: ajusta una sigmoide sobre los scores raw. Apto para distribuciones aproximadamente sigmoides.',
    '- **Isotonic regression**: monotónica no paramétrica. Más flexible pero requiere más datos para no sobre-ajustar.',
]

with open(REPORT_DIR / 'calibration_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print('calibration_summary.md guardado')

calibration_summary.md guardado
